In [67]:
!apt-get install -y cuda-cudart-13-0
!wget https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb
!dpkg -i cuda-keyring_1.1-1_all.deb
!apt-get update && apt-get install -y cuda-cudart-13-0

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
cuda-cudart-13-0 is already the newest version (13.0.96-1).
0 upgraded, 0 newly installed, 0 to remove and 67 not upgraded.
--2026-05-25 18:51:44--  https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb
Resolving developer.download.nvidia.com (developer.download.nvidia.com)... 23.213.43.207, 23.213.43.199
Connecting to developer.download.nvidia.com (developer.download.nvidia.com)|23.213.43.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4332 (4.2K) [application/x-deb]
Saving to: ‘cuda-keyring_1.1-1_all.deb.1’

cuda-keyring_1.1-1_ 100%[===================>]   4.23K  --.-KB/s    in 0s      

2026-05-25 18:51:46 (1.72 GB/s) - ‘cuda-keyring_1.1-1_all.deb.1’ saved [4332/4332]

(Reading database ... 122380 files and directories currently installed.)
Preparing to unpack cuda-keyring_1.1-1_all.deb ...
Unpacking cuda-keyri

In [68]:
!nvidia-smi

Mon May 25 18:51:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   76C    P0             33W /   72W |   20660MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [69]:
!pip install --upgrade pip
!pip install wandb hugginface_hub
!pip install vllm --extra-index-url https://download.pytorch.org/whl/cu129

ERROR: Could not find a version that satisfies the requirement hugginface_hub (from versions: none)
ERROR: No matching distribution found for hugginface_hub
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu129


In [70]:
from google.colab import drive
drive.mount('/content/drive')
 
import os
RESULTS_DIR = '/content/drive/MyDrive/resilient_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from huggingface_hub import login
import os

login()  # paste your HF token

import wandb
wandb.login(key=)  # paste your W&B token

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [72]:
import subprocess, shutil, os

# Disable FlashInfer JIT sampling (requires nvcc which is not available on this runtime)
env = os.environ.copy()
env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

vllm_proc = subprocess.Popen([
    "vllm", "serve", "google/gemma-4-E4B-it",
    "--max-model-len", "8192",
    "--gpu-memory-utilization", "0.90",
    "--dtype", "float16",
    "--trust-remote-code",
    "--port", "8000"
], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

for line in vllm_proc.stdout:
    line = line.decode("utf-8", errors="replace")
    print(line, end="")
    if "Application startup complete" in line or "Uvicorn running" in line:
        print("✅ vLLM server ready")
        break


(APIServer pid=28134) INFO 05-25 18:52:29 [utils.py:306] 
(APIServer pid=28134) INFO 05-25 18:52:29 [utils.py:306]        █     █     █▄   ▄█
(APIServer pid=28134) INFO 05-25 18:52:29 [utils.py:306]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.21.0
(APIServer pid=28134) INFO 05-25 18:52:29 [utils.py:306]   █▄█▀ █     █     █     █  model   google/gemma-4-E4B-it
(APIServer pid=28134) INFO 05-25 18:52:29 [utils.py:306]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=28134) INFO 05-25 18:52:29 [utils.py:306] 
(APIServer pid=28134) INFO 05-25 18:52:29 [utils.py:240] non-default args: {'model_tag': 'google/gemma-4-E4B-it', 'model': 'google/gemma-4-E4B-it', 'trust_remote_code': True, 'dtype': 'float16', 'max_model_len': 8192, 'gpu_memory_utilization': 0.9}
(APIServer pid=28134) INFO 05-25 18:52:31 [model.py:568] Resolved architecture: Gemma4ForConditionalGeneration
(APIServer pid=28134) WARNING 05-25 18:52:31 [model.py:2035] Casting torch.bfloat16 to torch.float16.
(APIServer pid=28134) INFO 05-25 18:52:

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY"   # vLLM doesn't enforce a key
)

response = client.chat.completions.create(
    model="google/gemma-4-E4B-it",
    messages=[{"role": "user", "content": "Explain PagedAttention briefly."}],
    max_tokens=512,
    temperature=0.7
)

print(response.choices[0].message.content)

In [73]:
# !git clone https://github.com/EvolvingLMMs-Lab/lmms-eval.git
# !cd lmms-eval && uv pip install -e ".[all]"

In [74]:
# !pip install decord
# !python -m lmms_eval --tasks list

In [75]:
# !python -m lmms_eval \
#   --model openai \
#   --model_args model=google/gemma-4-E4B-it,base_url=http://localhost:8000/v1/chat/completions,api_key=dummy \
#   --tasks mmmu_pro \
#   --batch_size 1 \
#   --verbosity=DEBUG \
#   --output_path ./results